# Analisis con Pandas y Kaggle (Core)


## Preparación del Entorno
* Asegúrate de tener instalado Pandas en tu entorno de trabajo.
* Descarga el archivo dataset.csv desde Kaggle. Elige un dataset que te interese y que no incluya visualización de datos. Algunas sugerencias pueden ser datasets relacionados con ventas, compras, productos, etc.

---

Se utilizara el dataset "Dirty Cafe Sales", ya que presenta amplia cantidad de datos, y con varios errores y null que requiere de su limpieza antes de poder realizar los analisis
Se puede encontrar en: https://www.kaggle.com/datasets/ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training

---

## Cargar los Datos
* Carga el archivo CSV en un DataFrame de Pandas.
* Muestra las primeras 10 filas del DataFrame para confirmar que los datos se han cargado correctamente.

In [1]:
import numpy as np
import pandas as pd

In [17]:
df = pd.read_csv("../data/dirty_cafe_sales.csv")

In [3]:
df.head(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,UNKNOWN,2023-10-28
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,NaN,In-store,2023-12-31


## Exploración Inicial de los Datos
* Muestra las últimas 5 filas del DataFrame.
* Utiliza el método info() para obtener información general sobre el DataFrame, incluyendo el número de entradas, nombres de las columnas, tipos de datos y memoria utilizada.
* Genera estadísticas descriptivas del DataFrame utilizando el método describe().

In [94]:
df.duplicated(subset=['Transaction ID']).sum()

np.int64(0)

In [4]:
df.tail()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02
9999,TXN_6170729,Sandwich,3,4.0,12.0,Cash,In-store,2023-11-07


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [6]:
porcNulos = (df.isnull().sum() / len(df)) * 100 #calcular porcentaje de nulos por columna
print(porcNulos)

Transaction ID       0.00
Item                 3.33
Quantity             1.38
Price Per Unit       1.79
Total Spent          1.73
Payment Method      25.79
Location            32.65
Transaction Date     1.59
dtype: float64


In [7]:
df.describe()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_9226047,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


---

* Dataset con 10 mil entradas
* Todas las columnas presentan datos "null" 
* Todas las columnas son de tipo "object" lo que hay que corregir
* Las columnas "Payment Method" y "Location" presentan la mayor cantidad de nulos con 25.8% y 32.7% respectivamente
* En columna "Transaction Date" aparece un valor Unknown como el mas frecuente con 159 apariciones
* Que no existan duplicados en el 'Transaction ID' me dice que podria repetirse la misma compra, pero que todas las compras son diferentes
* 
* De acuerdo al metodo "describe" con las columnas como objeto, habrian 10 items a vender, con 8 precios diferentes, 5 metodos de pago y 4 ubicaciones, lo que entrega un punto de partida para explorar

---

## Limpieza de Datos
* Identifica y maneja los datos faltantes utilizando técnicas apropiadas (relleno con valores estadísticos, interpolación, eliminación, etc.).
* Corrige los tipos de datos si es necesario (por ejemplo, convertir cadenas a fechas).
* Elimina duplicados si los hay.

---
* Evaluaremos las columnas que debemos pasar a numeros

---

In [8]:
df['Quantity'].value_counts()

Quantity
5          2013
2          1974
4          1863
3          1849
1          1822
UNKNOWN     171
ERROR       170
Name: count, dtype: int64

In [9]:
df['Price Per Unit'].value_counts()

Price Per Unit
3.0        2429
4.0        2331
2.0        1227
5.0        1204
1.0        1143
1.5        1133
ERROR       190
UNKNOWN     164
Name: count, dtype: int64

In [10]:
df['Item'].value_counts(dropna=False)

Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
NaN          333
ERROR        292
Name: count, dtype: int64

---

* Vemos que las columnas presentan valores UNKNOWN y ERROR
* Se reemplazara ambos valores con "null" para ayudar con la limpieza de datos
* se cambiara el tipo de las columnas Cantidad, Precio por unidad y total a numerico

---

In [18]:
#reemplazar todos los "Error" y "UNKNOWN" dentro del dataframe a null

df.replace('ERROR', np.nan, inplace=True)
df.replace('UNKNOWN', np.nan, inplace=True)


In [ ]:
#cambiamos los tipos de columnas a numericos.
paraCambiar = ['Quantity', 'Price Per Unit', 'Total Spent']
df[paraCambiar] = df[paraCambiar].apply(pd.to_numeric, errors='coerce')

In [26]:
porcNulos = (df.isnull().sum() / len(df)) * 100 #calcular porcentaje de nulos por columna
print(porcNulos)

Transaction ID       0.00
Item                 9.69
Quantity             4.79
Price Per Unit       0.54
Total Spent          5.02
Payment Method      31.78
Location            39.61
Transaction Date     4.60
dtype: float64


---
* ARRIBA: Vemos como al cambiar 'Error' y 'Unknown' a null, se muestra la cantidad escondida de datos faltantes en el dataframe. Ahora se observan porcentajes mas cercanos a la realidad
<br />
<br />
SIGUIENTE: 
* Se creara un diccionario que establezca la relacion entre item y precio
* se usara el diccionario para rellenar los datos faltantes de "Price per Unit"
---

In [13]:
#hare un diccionario con los items y sus precios
precios = {
    'Juice': 3.0,
    'Coffee': 2.0,
    'Cake': 3.0,
    'Salad': 5.0,
    'Sandwich': 4.0,
    'Smoothie': 4.0,
    'Cookie': 1.0,
    'Tea': 1.5
}

In [ ]:
#usare los nombres de los item para rellenar con el precio correspondiente
df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Item'].map(precios))

In [33]:
df[['Item', 'Price Per Unit']].value_counts(dropna=False, normalize=True)

Item      Price Per Unit
Juice     3.0               0.1171
Coffee    2.0               0.1165
Salad     5.0               0.1148
Cake      3.0               0.1139
Sandwich  4.0               0.1131
Smoothie  4.0               0.1096
Cookie    1.0               0.1092
Tea       1.5               0.1089
NaN       3.0               0.0234
          4.0               0.0213
          5.0               0.0122
          2.0               0.0119
          1.0               0.0117
          1.5               0.0110
          NaN               0.0054
Name: proportion, dtype: float64

---
* Considerando que tengo 4 productos que comparten 2 precios diferentes, pero las proporciones de nulos que quedan todavia mantiene proporciones similares entre todos los items (aprox. 0.01 pro producto), decidi eliminar todas las filas con nulos en la columna item


---

In [36]:
df2 = df.dropna(subset=['Item'])

In [37]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9031 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    9031 non-null   object 
 1   Item              9031 non-null   object 
 2   Quantity          8611 non-null   float64
 3   Price Per Unit    9031 non-null   float64
 4   Total Spent       8579 non-null   float64
 5   Payment Method    6154 non-null   object 
 6   Location          5448 non-null   object 
 7   Transaction Date  8613 non-null   object 
dtypes: float64(3), object(5)
memory usage: 635.0+ KB


In [49]:
porcNulos = (df2.isnull().sum() / len(df2)) * 100 #calcular porcentaje de nulos por columna
print(porcNulos)

Transaction ID       0.000000
Item                 0.000000
Quantity             0.221459
Price Per Unit       0.000000
Total Spent          0.221459
Payment Method      31.856937
Location            39.674455
Transaction Date     4.628502
dtype: float64


---

Si tengo el precio por cantidad, con tener solo la cantidad o el total gastado, podria obtener el valor faltante

---

In [39]:
df2.sample(20)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
5756,TXN_9640865,Cookie,2.0,1.0,NaN,Cash,Takeaway,2023-03-18
7213,TXN_2566299,Juice,2.0,3.0,6.0,Cash,NaN,2023-08-20
7968,TXN_2457961,Coffee,2.0,2.0,4.0,Credit Card,In-store,2023-06-08
6896,TXN_1887555,Juice,2.0,3.0,6.0,Cash,In-store,2023-01-08
973,TXN_3101575,Coffee,NaN,2.0,4.0,NaN,Takeaway,2023-12-03
8862,TXN_6016707,Juice,1.0,3.0,3.0,Cash,In-store,2023-06-27
8813,TXN_5542500,Sandwich,3.0,4.0,12.0,Credit Card,NaN,2023-07-19
2974,TXN_8571836,Juice,3.0,3.0,9.0,Cash,NaN,2023-12-27
2915,TXN_1522430,Smoothie,3.0,4.0,12.0,Credit Card,NaN,2023-12-06
144,TXN_4811133,Salad,2.0,5.0,10.0,Cash,NaN,2023-08-11


In [40]:
# Vamos a rellenar Cantidad
cond_cant = df2['Quantity'].isnull() & df2['Total Spent'].notnull()
df2.loc[cond_cant, 'Quantity'] = df2['Total Spent'] / df2['Price Per Unit']

In [41]:
# Vamos a rellenar el Total
cond_cant = df2['Total Spent'].isnull() & df2['Quantity'].notnull()
df2.loc[cond_cant, 'Total Spent'] = df2['Price Per Unit'] * df2['Quantity']

In [43]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9031 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    9031 non-null   object 
 1   Item              9031 non-null   object 
 2   Quantity          9011 non-null   float64
 3   Price Per Unit    9031 non-null   float64
 4   Total Spent       9011 non-null   float64
 5   Payment Method    6154 non-null   object 
 6   Location          5448 non-null   object 
 7   Transaction Date  8613 non-null   object 
dtypes: float64(3), object(5)
memory usage: 893.0+ KB


---

Luego de corregir los datos faltantes en Cantidad y Totales, vamos a revisar Payment y Location

---

In [44]:
df['Location'].value_counts(dropna=False)

Location
NaN         3961
Takeaway    3022
In-store    3017
Name: count, dtype: int64

In [45]:
df['Payment Method'].value_counts(dropna=False)

Payment Method
NaN               3178
Digital Wallet    2291
Credit Card       2273
Cash              2258
Name: count, dtype: int64

In [50]:
df[['Location', 'Payment Method']].value_counts()

Location  Payment Method
Takeaway  Digital Wallet    744
In-store  Cash              702
          Digital Wallet    695
          Credit Card       681
Takeaway  Credit Card       669
          Cash              664
Name: count, dtype: int64

In [51]:
df[['Location', 'Payment Method']].value_counts(dropna=False)

Location  Payment Method
NaN       NaN               1294
Takeaway  NaN                945
In-store  NaN                939
NaN       Credit Card        923
          Cash               892
          Digital Wallet     852
Takeaway  Digital Wallet     744
In-store  Cash               702
          Digital Wallet     695
          Credit Card        681
Takeaway  Credit Card        669
          Cash               664
Name: count, dtype: int64

---

Considerando que las columnas Location y Payment Method tienen un 40% y un 32% de nulos respectivamente, lo logico es eliminarlas del dataset.
Sin embargo, con ayuda de Gemini, me mostro una manera de poder utilizarla rellenando los valores usando las proporciones de cada item, considerando que son bastante similares entre ellos.

---

In [ ]:
def fill_proportionally(df, column):
    # Obtener valores no nulos y sus proporciones
    counts = df[column].value_counts(normalize=True)
    # Identificar dónde están los nulos
    is_null = df[column].isnull()
    # Generar valores aleatorios basados en la distribución observada
    df.loc[is_null, column] = np.random.choice(
        counts.index, 
        size=is_null.sum(), 
        p=counts.values
    )
    return df

df2 = fill_proportionally(df2, 'Location')
df2 = fill_proportionally(df2, 'Payment Method')

In [ ]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9031 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    9031 non-null   object 
 1   Item              9031 non-null   object 
 2   Quantity          9011 non-null   float64
 3   Price Per Unit    9031 non-null   float64
 4   Total Spent       9011 non-null   float64
 5   Payment Method    9031 non-null   object 
 6   Location          9031 non-null   object 
 7   Transaction Date  8613 non-null   object 
dtypes: float64(3), object(5)
memory usage: 893.0+ KB


In [ ]:
df2.sample(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
5753,TXN_9655021,Coffee,3.0,2.0,6.0,Credit Card,In-store,2023-04-22
7259,TXN_8159772,Cake,1.0,3.0,3.0,Credit Card,In-store,2023-10-02
4555,TXN_9082461,Cake,5.0,3.0,15.0,Digital Wallet,In-store,2023-05-30
5871,TXN_7648238,Cookie,1.0,1.0,1.0,Cash,Takeaway,2023-05-12
5836,TXN_5229351,Coffee,3.0,2.0,6.0,Digital Wallet,Takeaway,2023-08-14
3213,TXN_1399696,Cake,5.0,3.0,15.0,Cash,In-store,2023-04-08
2942,TXN_4462052,Cookie,1.0,1.0,1.0,Cash,In-store,2023-10-12
4940,TXN_4637438,Salad,2.0,5.0,10.0,Digital Wallet,Takeaway,2023-04-07
2062,TXN_4868913,Cake,4.0,3.0,12.0,Cash,Takeaway,2023-06-30
5170,TXN_5519424,Tea,2.0,1.5,3.0,Digital Wallet,Takeaway,2023-09-23


In [ ]:
#Como no son los valores originales, vamos a cambiar los nombres de las columnas Location y Payment Method
df2 = df2.rename(columns={
    'Location': 'InferLocation',
    'Payment Method': 'InferPayment'
})

In [ ]:
#Cambiamos el tipo de la columna Transaction Date a fecha
df2['Transaction Date'] = pd.to_datetime(df2['Transaction Date'])

---

Como las filas estan desordenadas, no puedo basarme en informacion anterior o posterior para recuperar esos datos
Se eliminan las filas de "Transaction date" que tengan nulos, junto con las de quantity y total spent

---

In [80]:
df2.dropna(subset=['Transaction Date', 'Quantity'], inplace=True)

In [81]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8593 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    8593 non-null   object        
 1   Item              8593 non-null   object        
 2   Quantity          8593 non-null   float64       
 3   Price Per Unit    8593 non-null   float64       
 4   Total Spent       8593 non-null   float64       
 5   InferPayment      8593 non-null   object        
 6   InferLocation     8593 non-null   object        
 7   Transaction Date  8593 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 604.2+ KB


## Transformación de Datos
* Crea nuevas columnas basadas en operaciones con las columnas existentes (por ejemplo, calcular ingresos a partir de ventas y precios).
* Normaliza o estandariza columnas si es necesario.
* Clasifica los datos en categorías relevantes.

---

Vamos a crear 2 columnas para entender mejor la variacion en compras: Mes y Quarter
De acuerdo al dataset, los registros ocurren solamente dentro del año 2023

---

In [86]:
df2['Month'] = df2['Transaction Date'].dt.month_name()

In [87]:
df2['Quarter'] = df2['Transaction Date'].dt.quarter

In [88]:
df2.sample(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,InferPayment,InferLocation,Transaction Date,Month,Quarter
7544,TXN_5896141,Smoothie,5.0,4.0,20.0,Credit Card,In-store,2023-03-08,March,1
3836,TXN_5687012,Salad,2.0,5.0,10.0,Cash,Takeaway,2023-03-12,March,1
1668,TXN_3450160,Cake,4.0,3.0,12.0,Credit Card,In-store,2023-10-23,October,4
4559,TXN_3042768,Cookie,3.0,1.0,3.0,Digital Wallet,Takeaway,2023-09-07,September,3
2900,TXN_7003185,Tea,2.0,1.5,3.0,Cash,Takeaway,2023-06-03,June,2
8436,TXN_3641356,Juice,3.0,3.0,9.0,Credit Card,In-store,2023-07-26,July,3
6687,TXN_8211303,Salad,5.0,5.0,25.0,Credit Card,In-store,2023-08-26,August,3
2346,TXN_4411172,Smoothie,4.0,4.0,16.0,Cash,Takeaway,2023-07-09,July,3
6914,TXN_6425954,Sandwich,2.0,4.0,8.0,Digital Wallet,In-store,2023-03-28,March,1
3952,TXN_9157460,Cake,4.0,3.0,12.0,Digital Wallet,Takeaway,2023-01-23,January,1


---

 Vamos a crear una categoria para separar los items vendidos en Dulces, Salados y Bebestibles
 
---

In [ ]:
# Creamos diccionario con las categorias a utilizar
categoria_dic= {
    'Cake': 'Sweet',
    'Cookie': 'Sweet',
    'Sandwich': 'Savory',
    'Salad': 'Savory',
    'Juice': 'Beverage',
    'Tea': 'Beverage',
    'Coffee': 'Beverage',
    'Smoothie': 'Beverage'
}

#aplicamos el diccionario
df2['Category'] = df2['Item'].map(categoria_dic)

In [95]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8593 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    8593 non-null   object        
 1   Item              8593 non-null   object        
 2   Quantity          8593 non-null   float64       
 3   Price Per Unit    8593 non-null   float64       
 4   Total Spent       8593 non-null   float64       
 5   InferPayment      8593 non-null   object        
 6   InferLocation     8593 non-null   object        
 7   Transaction Date  8593 non-null   datetime64[ns]
 8   Month             8593 non-null   object        
 9   Quarter           8593 non-null   int32         
 10  Category          8593 non-null   object        
dtypes: datetime64[ns](1), float64(3), int32(1), object(6)
memory usage: 772.0+ KB


In [90]:
df2.sample(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,InferPayment,InferLocation,Transaction Date,Month,Quarter,Category
1944,TXN_2399870,Salad,4.0,5.0,20.0,Cash,Takeaway,2023-04-08,April,2,Savory
3635,TXN_6177081,Cookie,1.0,1.0,1.0,Cash,Takeaway,2023-07-26,July,3,Sweet
7709,TXN_4472969,Juice,1.0,3.0,3.0,Credit Card,In-store,2023-09-15,September,3,Beverage
3632,TXN_9362728,Sandwich,1.0,4.0,4.0,Credit Card,In-store,2023-12-26,December,4,Savory
9042,TXN_1666190,Smoothie,4.0,4.0,16.0,Credit Card,Takeaway,2023-07-28,July,3,Beverage
9630,TXN_2862856,Cake,5.0,3.0,15.0,Digital Wallet,Takeaway,2023-02-01,February,1,Sweet
1869,TXN_2699583,Sandwich,5.0,4.0,20.0,Digital Wallet,In-store,2023-07-26,July,3,Savory
7841,TXN_9454387,Salad,2.0,5.0,10.0,Cash,Takeaway,2023-01-02,January,1,Savory
9930,TXN_4428252,Sandwich,3.0,4.0,12.0,Cash,Takeaway,2023-10-31,October,4,Savory
9097,TXN_9904214,Sandwich,5.0,4.0,20.0,Cash,Takeaway,2023-03-13,March,1,Savory


## Análisis de Datos
* Realiza agrupaciones de datos utilizando groupby para obtener insights específicos (por ejemplo, ventas por producto, ventas por región, etc.).
* Aplica funciones de agregación como sum, mean, count, min, max, std, y var.
* Utiliza el método apply para realizar operaciones más complejas y personalizadas.

In [ ]:
#revisar los ingresos que generan las categorias creadas
df2.groupby('Category')['Total Spent'].sum()


Category
Beverage    34278.0
Savory      29643.0
Sweet       12910.0
Name: Total Spent, dtype: float64

In [ ]:
#revisar el detalle de ventas agrupadas por quarter. Se revisa ingreso total, cantidad de ventas y cantidad de items vendidos
df2.groupby(['Quarter']).agg({'Total Spent': 'sum', 'Transaction ID' : 'count', 'Quantity' : 'sum'})

,Total Spent,Transaction ID,Quantity
Quarter,,,
1,19170.0,2134,6501.0
2,19300.5,2124,6449.0
3,18731.5,2138,6365.0
4,19629.0,2197,6673.0


In [112]:
#Revisamos la cantidad de compras por categoria el dinero que ingreso por esto
df2.groupby(['Category']).agg({'Total Spent': 'sum', 'Transaction ID' : 'count'})


,Total Spent,Transaction ID
Category,,
Beverage,34278.0,4311
Savory,29643.0,2168
Sweet,12910.0,2114


In [ ]:
#revisamos las compras totales por mes, junto a la cantidad de compras que existieron en dicho mes
df2.groupby(['Month']).agg({'Total Spent': 'sum', 'Transaction ID' : 'count'})

,Total Spent,Transaction ID
Month,,
April,6464.5,696
August,6348.0,715
December,6436.0,710
February,5953.5,649
January,6690.0,742
July,6175.5,717
June,6616.0,733
March,6526.5,743
May,6220.0,695


In [ ]:
# Se buscan los mejores meses, donde hubo un mayor ingreso por cantidad de ventas
IngresoPorVenta = df2.groupby(['Month']).apply(
    lambda x: pd.Series({
        'Ingresos': x['Total Spent'].sum(),
        'CantVentas': x['Transaction ID'].count(),
        'Tasa': x['Total Spent'].sum() / x['Transaction ID'].count()
    })
)

C:\Users\jr_ca\AppData\Local\Temp\ipykernel_27168\2232611086.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  IngresoPorVenta = df2.groupby(['Month']).apply(


In [124]:
IngresoPorVenta

,Ingresos,CantVentas,Tasa
Month,,,
April,6464.5,696.0,9.288075
August,6348.0,715.0,8.878322
December,6436.0,710.0,9.064789
February,5953.5,649.0,9.173344
January,6690.0,742.0,9.016173
July,6175.5,717.0,8.612971
June,6616.0,733.0,9.025921
March,6526.5,743.0,8.783984
May,6220.0,695.0,8.949640


---
* Los bebestibles son la categoria que genera masyor ingreso. Eso es esperable, ya que tiene 4 items, mientras que dulce y salado solo tienen 2.
* Los items salados, a pesar de tener 2 items, genero un 86% del ingreso que generaron los bebestibles con la mitad de las transacciones del 2023.

* Los meses con mayor cantidad de ingresos es Octubre, Enero y Junio.
* Los meses con mayor cantidad de transacciones es Octubre, Marzo, Enero.
* El mes de Abril, presenta el mayor ingreso por venta con 696 ventas a un promedio de 9.3 dolares aprox.

* Durante el mes de Octubre se produce el mayor ingreso y la mayor cantidad de ventas.
*  El Q4 del 2023, es donde ocurre el mayor ingreso, la mayor cantidad de ventas y la mayor cantidad de items vendidos.

---

## Documentación
* Documenta claramente cada paso del análisis, explicando qué se hizo y por qué se hizo.
* Asegúrate de que el código sea legible y esté bien comentado.